#Data Quality Snapshot
###Read pipeline even log and save quality rules metrics

In [0]:
CREATE OR REPLACE TABLE workspace.food_safety.gold_dq_expectations AS
WITH exp AS (
  SELECT
    timestamp,
    explode(from_json(
      details:flow_progress.data_quality.expectations,
      'array<struct<name:string, dataset:string, passed_records:bigint, failed_records:bigint>>'
    )) AS e
  FROM event_log(TABLE(workspace.food_safety.silver_recall_events))
  WHERE event_type = 'flow_progress'
    AND details:flow_progress.data_quality.expectations IS NOT NULL
)
SELECT
  e.dataset,
  e.name                  AS expectation,
  SUM(e.passed_records)   AS passed_records,
  SUM(e.failed_records)   AS failed_records,
  ROUND(100 * SUM(e.failed_records)
        / NULLIF(SUM(e.passed_records) + SUM(e.failed_records), 0), 3) AS failure_pct,
  MAX(timestamp)          AS last_run
FROM exp
GROUP BY ALL;
 
SELECT * FROM workspace.food_safety.gold_dq_expectations ORDER BY failed_records DESC;
